# master9 — Parametric inertial particle PINN (GPU)

این نسخه با نسخهٔ قبلیِ کنترل نظارت‌شده فرق دارد: **هیچ برچسب فرمان و هیچ مسیر حل‌شده‌ای وارد آموزش نمی‌شود.** آموزش از باقیماندهٔ معادلات سیال و ذره است؛ حل مستقل Radau فقط بعد از آموزش اجرا می‌شود.

ورودی‌ها: رینولدز، سرعت متوسط سیال، چگالی سیال، محل رهایش، دو مؤلفهٔ سرعت اولیهٔ ذره و تحریک مغناطیسی. ویسکوزیته از `eta=rho*U*(2R)/Re` به دست می‌آید و ترکیب نامعتبر رد می‌شود.

محدوده‌ها: eta=0.001..0.006 Pa.s، U=0.005..0.03 m/s، rho=950..1100 kg/m³، X0=-0.7..0.7، Z0=-2..1.7، vx0=±0.001 m/s، vz0=0..0.06 m/s، u=±0.5. شعاع ذره 500nm و کانال 75µm ثابت‌اند.

معادلات: جریان کاملاً توسعه‌یافتهٔ نیوتنی در کانال استوانه‌ای، میدان تحلیلی دو منبع آینه‌ای و حرکت ذره با درگ استوکس و جرم افزودهٔ نصف جرم سیال جابه‌جاشده. گرانش، نیروی تاریخچهٔ Basset، برهم‌کنش دوطرفه و نویز حسگر لحاظ نشده‌اند. گذرای زیرمیکروثانیه‌ای فقط در محدودهٔ این مدل ساده اعتبارسنجی می‌شود.

ساختار شبکه شرط مکان و سرعت اولیه را دقیقاً اعمال می‌کند و لایهٔ گذرای نمایی را از بخش کند حرکت جدا می‌کند. PINN نگاشت گام زمانی کوتاه را می‌آموزد و برای مسیر بلند تکرار می‌شود. کنترل فرمان، حل معکوس مقید با همین PINN است، نه شبکهٔ دارای برچسب جواب.


In [ ]:
import os,sys,io,base64,zipfile,json,time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'GPU was requested but is unavailable; do not silently train on CPU.'
ROOT=Path.cwd()/'master9_run'
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('UEsDBBQAAAAIAElpY109uSfnSwAAAE4AAAATAAAAbWFzdGVyOS9fX2luaXRfXy5weRXIQQ5AQAwF0L1TNN2z5wYWJq7Q0PCTKulUxO3FWz5mnqSmRj/QJSGHZmAhuEZC7L/EYkrzWAo9yJ1CK9ZbrD3dXsoQOHzrmLn5AFBLAwQUAAAACAC5aWNde+IGes8FAAC3DAAAEgAAAG1hc3RlcjkvY29udHJvbC5wea1XW2/bNhR+z68Q8kLKpTXJrZPVDgcM2ID1pSuyYi+BITASZXOhRIWkXNtF//sOL7blNN3TDAMmD8+F58LvHF9fX//ZSdHxpFKdsZrBsk5Et+XaCNUlqknshieaG1EPTE6PHJ8+fPy4TDrlxKxWMpHskUuTXV9fXzVatUnP7EaKx0S0vdI2+QTbcFAzyyrJjOHmeMhMLSp7FXf/GNUd193Q9ns4T7o+SJtK9PtM9Va04sCPCh417+xz4Mj6zd6I6qR8EslbJgXYPsmAJ50RFpy8uqp5kzRKSvUFt6rmkvRME9b2Utih5jTLZ+RQ8q6mRfaOtGxXGst7Q2/zPF1cJfAB/pMBnC6fyZY6kujAAJOOoulzVql+D+utptvj+pDT54ditdy533zllammMdzSrs8s6zYYH/I32dt0Mkv9qburZXrNLT6kC83toLtkl785XXeCz5JRcBpUBgVafTH0IZhqlE6cL5DxBOKx5vjkXfTMfUSTsEeD3QXTX7LbBITcpX8p4lqHzc9u4xl15FxAYtjTSY+7OtdaaTyMtLtPt7P0nJCYBIgiGVwq0gve6DKIgJVpDIXbFaszo1RkI2gwNs3mKQlLWI29kmqyEXc0Xwy0kYpZHAoJe2YCcgS+O6skLfj0fXqWhVLnlx5UrKt9+iG2TjDPQHa1HGgrOnw+JE98TyVrH2uWDAsXrGNARtpdhjLW91Bx+MElY/LH5PNfxMV/cv83cbGf/Eai526XOvJAfNw9h75k0ZFnFCBXoj+OONlyqSph9/SzHvhICk40VRpeMPdlgj3lZZogtL5AqH81l4G6cA47HW+K9P9zcHkuOeaeENOa7bEzGq4H4Y65hkModuxywB4WZL6aup/bVZpOCn4TH5sWjS13r0sUQWJ+lFgG7sPr3LPAfXOhX3OHRfQrgvCxlltAXbQIcIhdRAm652jhoOSeE+Tepuhq7mIHdVpC/qon0a1LX0Pl0KIFrMgp3F7AZ7aMKduVtWgaDlVecc8fHSTfsx5eZz2M1FcKMAcuXaPFo1ISj3KeQMVHYAiEaVaANx5Z0ELyDrM0GIX4lANahJiNA/YOIpV+G4XpAaIEbaNGK+rNHanna6y8WYjBXXY79+vo31023h7uinl+TIAHE0aCstAKat4qE5+EGmw/2AhXsXW0zPZSWWhvnnreZoPhGP26XqP0df6s37uV62i9tAHtvQHqGiSOxpaNWBO2AygBpswMj07G4BmZETgx0PgoLgpym6bLcG1Dv4ZAVQzaKpRTpwB2mESLT6e6wukoc9DpStNzXpcbsd6UW2EqBUiwvxBoOeug9+Vzwi1zixsCbQoSOXdPEDrXFDAfHuEWqHBakC3Q8rEZr/yelw0zttRccrjeaybeHi0URG8ULYo8d6YuLN1GS9OTqSy/GRszG8gtePQfdmZHO++8nffzPHg0G9m5OXsEngc7+TwWouuXbEdwB4o95LnGeRA9dtnKGsks8SnIhOUthHzU5qDE6MsZ45S/B6dwRfWy16KzGMWpChFvSJNGDmbzAosBXwzbcruzsW5+ao5y5Vcn9y2rzBYRRsAajEsQCIoIIvBcWkAPQxEiG85qR4ZxCvCc7Eo315UtTDtxEQAWcIPZQDoAcQDGe1azwXMeV2PWQANedL4v22WujiMYTiDHOXHrtyuHhwQ9TaeI+DGSos9eF0qXPxIqglBkj5nWovKD6Qu5m5HcPBpbnEx9OAMqJJNrN7Al/v6Xd4f5CVthJac+J7sofkhw26aI7ON+B3tRwcQJ1+EmDVeRfO3aXQPZ8a/3doz+5gExKcsTtsEG6zPW+ZrTfkAL7G7QHLgrruUx8ae8w4QBDG6CRmn2RUPOgQQF4ihZDcO0iahpiPAu01mcOwBYMgvP1ZaS7UEt9ijkKwx+vzMEvg0STPXdGpG6F7S4iYgKvb8DhHvVtwX8ezA8uR86V3C/+8kHOeQ7/Y1gVcV7GF4rnjRMSBC6wOmo9epfUEsDBBQAAAAIAM5pY10bIYpY8AMAAEAJAAARAAAAbWFzdGVyOS9tb2RlbHMucHm1Vdtu2zgQffdXEN4XUqZpSYmTRoYeFlgUXaANgjToSyAIjETFwtKUTFJO0q/foe7Jpt0LsAIsU8O5nJk5HC6Xy1thyrzhcm01L5XIUSGbMkdc5UhWGZcIhNqWsChk9bQ+8Brd/H59bRi6rpDkD0JKMDKVbGxZKcOWy+WiPNSVtshWOtsvCl0duiXq5Up1QlbvX0yZmUHuLTLJjUFfPt9gpdiXKm+kINECwZOLAqVpqUqbptgIWVBFq8bSpzK3+/jqguaihsVZr+4e09RCY8JGM7KT/EVoE98no1JRaZRCjkhz9Shw64VEnR4Tz1aoHDswn6EMXGPVBSQURHdc7TEhZKfiVjj67K15XTvruTEgBn0HnylhY9j6Ko6NUK6+2OvsyJgvYHviOu/SfSaRFrbRCg3m+Jn0BfsInXFN+fuqkeidqox4XOEDGtCzkIY/gHGk9QgEB+sj8UY4oXdcBwOkGw6kyaT4Z7Co49bPW9eGcWqxe73GfDXAHTpau46OFqzmmh+EheJiEtVMQ81LLUz6qHme4o9cGjGlm1Vai8yRuYNm6NGnJ582LvMxysnA2RBxy2sGLDGVxvfM9wPv7uvm9htl/oVb/ZYAMU9lJmLDugXN7Uvdfrr/CXap6t5bxi2GYpp1AJE3byJc0pD9xScBfJsOEW02bEuFyqpcpG3eBteE0GCK9JZGEHnKHooEFm8Tp0ZklcrjrlRTFVTcepmXbFatSc3MUrvvlryxlas/cy+s7iNaJsw0B0wgcKYFt8L1p97Hd7oRFEDDeJpJyL2ftL0up9MbkmSeqDJD3O9CVyaV5R8Cq1nJC9TlNSX02uzHcM3/gdfGnzyzE7WJ4ZdWBTRuV/ix3ht8HNrQU3oyGjgonmu8thuwnPaO8dFfWa/wVyD2YHvQPASDLvHwyV8XPgE966mJ37GzGjdDt9mpmM2nUesX9Kt8FA9wcwD35AvKuMqERMaWRYEgTqW62dbRCj0IKICAE3ao4bZQj8juBdCxu4DY6PWBS+fHIQg9bB14AtExhF+dey4VS1ocbVodKPjyvPAtx4/0RHtv74+z9w/3f2rEEPLfVfyn52fh8Bp+qCXwtj/0EFrk6aOAW5mD9/gaatxD5z0VgFzu0qJXZBgUvelk9dpJlwQwNm4n2IXnceC3HwXJ7iC4ctLtivnhtpUHUZjs9L6Kr7b+Ktj6rTCMzrpL9dUUAy3PefBC7/bbBgJQ90VBPE6k41x/zS5XATtvPZ5F5wldh6szdtl+n0fbZLQ6za3ADGADQD9sNbfRRUJmc9jJLqLLpB3Io4smXrPtyu1dRh+SnYnd8kN0lby9H9rTu5hPTup4BR1a/AlQSwMEFAAAAAgAiWljXZPiKpqfBQAAAwwAABIAAABtYXN0ZXI5L3BoeXNpY3MucHl1Vm1v2zYQ/u5fQWQfLDG03lwnjV0N7ZBuGbB1gd0GwwzBYGU6Fqy3ipJjuch/3x0pyXLSGrBEHu/1uTueLi4uFn+SnBc8EaUoJCNplq6jRKQyylIekygVRRnBYlFmOyFJkq1FTHi6hpO1yAU80pJkBQ9jYV1cXAw2RZaQNS+BwKUEiSjJs6I8kRiX6ygsBw09rZK8JlySNG9JZVaE28H8wb+eiNHVjNz6lvNmRr7MP/4OK3cyI58X/q1teBRJ5ozM7/2J44jRDSzv/oGN5zjWjPz9xfHf0DS38oi6YnQ9+G3hAw3WV9RyPEo91HELb3Pw4a/7uw/+mBqtwPye0rE9Nm2QGNyBXW8ymH/49MfHhf99KEq+uudyOF1ajuMyeFwFbJgInq6SVUOeAHkM1GKbrXaPq2QM5JuJYzHXBe8CNiDNb1iIWHApVv8Cx8i6ZtY1ijXE/5DogZSiRmmE2VjtD42hUeuA29fYsR0bNjBrKR8rJQO+TYLnwfsuKQP1JPddIUyVMoxouokzXirgFQ1i70jO2Lu+unHh6TnueOK+URwQccMBkTqWoh2chtTsj+0eAnirKPuXLPuOB4BUlPd5keVQj7XarcWGzIUhRbwxp4UoqyIluLHAPlULdJ96dP5gqy14/hM1ADY/8BJKfiXPFXrUwJK6tCa01W2q2oDauaGtWlPrVSACftts3anGflgVwghjyeaCoUsKSwaaND6M0t2TOe2yF22gCaExciuSG8ykMObChB6DaN/5zrTgkRTkgceV+FgUWWEM54IklSzJV0HyTELq92JodvqaSMABQ1nHh7IOfwbO45ueoAInlUPWnscRFIgwzC6YjqQx6kzAHVBEQvY7o4Wm3xZdUs66osX1VL8vu0JxHBx7/nDWF4p8dOzb132hjqCi2KtW0CdH57mztgFkUyh7doA7rQ3FAtgTafRixF+csW3k62tgiTLB2fEPMndQiTu8i7MRXEGup3a/bqNLtXudy83wO+p9JllVymgtSAksqViTdZbAgnxXLjy/Ti/G1eWphPsblCGNrcU+CoU/DPNqyNZlnQtfXbCW6q2x1wuxUaWPGx3LpUIMquKUvTZjQdBq169GvXqeiqbJwHlfAUa8KHhtLHvJ7RIamOwFA+SSfl50TJBA3ALjYIA2sOyyjZF3+vPllHnTcUBx4U69QBc37pypG2ihXJ4L9Zq9le81e2cDLJtKwSNk5FyDGiP0t4USuWpGyUkQdOmhBb5oFSINYaKu1ASWqKi9Y/1OSlF+geYnkSSyyvM4gnoQB3iHURnXM3IaxPtIhngD1Mibir0oiORJHouzeR3XJIO8lPqePct6yEvD0Ms4ezRavGzremLaHjNaOOHadpEEL2+iyQjXyIU5adpwwlwdIHRXKIxvrGpCw6JuCvBpK+BirN45MNzYyLV0pLXvXiIT/dbYxrzdzo7+t8b0DELwjZrWl0d6NGFK/ySKkUdrippsEGCwO+ICHaP8qzQq7V+xlegdyxn0w5P/KUtF4+lm45+cn+38U7bV8ZPvjjoXqYfNjxoQeFRCRAydjRSjx8Ua4dfe7uhm0/CxFmRblcrT5U5BoBmQ3oGrv7tWXQy8aHw/MMBrprB2LXQNUNYeAc6zDuKDUtxGCxD1ug7Pa3YMTLuPdQsdiuz8H5Y7eIEX/4ti70V9MqKCdgJ0XN0sOmDDHR3oweyF7QbBWcCyFDlEvNcxs61/1+YMvz5lGOW1FaWleCxgWLVfoDKL92IV7XPdYrn00Wx/9NufF6e5XaVGyer+hVWslvXSmwbM6OFeLyEhDfYjdWzaoFsPBjDpd2YNUMmWDtsGTCuDAAKmPxf84ZyvOVzQRQkinhi9ZRxXOCUcDV4zXECdJaswFFI202NepSV8sev5gccJnPFHcYY50tFV6LOA6Y03xc3gf1BLAwQUAAAACADOaWNdrnPaH64FAAC5DQAAEAAAAG1hc3RlcjkvdHJhaW4ucHmtV9tu2zgQfc9XEFgsRDkMI6fNYleGHoJub0ARFO32KTAERmJs1hJJk5Qdt+i/75CUHDmXpg91EVklOcO5nDMzvjGqRZq5ZSOukWi1Mg59hP8e9e9frZLEiZYTZYc12bV6h5hFUhOnTLU8uvFaaKtq3thBy+ToqOY3yBkmJFad050jN43altZxbYsXWZYRzYwTVcP7temZX7xmrloWZ9nLv4nh604YXi50V/xnOp7mRwg+UVvh7ew1p7P4TdtVLQwGvVw6G2QIvxXWlWoVNQQF4gaNVCMmaySVQ8EZWnU1o8KWbMNEw64bjtMcnLAcfeqkD8VrY5TByQV6+/ELMp1EW4iF18fBjRpddw69+vLvBRIWdXKvhSbx7ppvRMWLxF+TeEuevBVBNDlKKt0lg9W9cBEW8yhpuSshJaVbGs5qi8/iPXGzZbJjTWk5r/E/2Yt0JjU14K9q6X4tHDfRt+J7Eu9I8vhNksXdVcHIBdwX90rJWo6z9NCy4Fcw/VJJkA+iSW6dwVFLWW64sULJskxJ4s1IcrCDJCHzSR6+SAJR4E3D6xIg9JVXICu4TfJstFMp6Yxq+lVl4SW5aTpRo1a1gICuhbxYUUMIZkhIDmhjDfrs1Irb/Q46RktmaqSVFQ6sOt3wRlXC7dD7V8mPEBxthHTYc4HWAH6L+2ilgOjOLiOyZtYBnAu/Tv0Dx8B6zBdv4PHx/eUlTqlTOMYKMKvheAgJvImWXtSsxf48BQBDaB2ECaekMQXNsmlU16hFcTWPmpVBnjjgGYKcLji+41fPFP8pSfynC8taDcCK8R2MWPcmeFTgfm+a9ttFD4KeLbZcGFaPiOQ/28Jfi9dEp7PtoI11Tvmz1D/wltquBU/WpAKMOu7V6D5sV9n8TtX6Cfn1LyoAE4v1BNQcb9fH0/2yh0aBYXMCfyltOZP4zoGFEXV/bSOk1awCUJMp+ev8MArpVU48pO+ug+zf9pIAUs2/KVGHJGCvk+irfDqn/Fb70IK2k2nqdWRz4rfDW3poo38cT7MJ9ppP6Hk6mZztTwBI6DduVEgCTmf+ML1m1WoL8MUBTtQnf+QaENOv/HmeZUWR5RHHibcwIX7DV2TmcNBUc8eqJU4PQQ14o0xrDh5cPSc4f3Cxhyaewu3kDB4jUA74XfgT3vCAePBMddrmi6ukMcl8UtAXew5RvmHNwCmQDLofkCWH8jbGaonfMChFUSw0KGgbsekEPnoFz3MyCFLJ3U+Iaaslr7uGH0g3phzWDX0FBUbyCyk5A5wtPnzCcOheDyQQy7IVsjjjJ+fpEzQ/FBlF1QJBNqR7gur7Y3+g97JqupqDShFKouENu2W++qGG7bhBHZi68MWGGeGWrahY0+xQ4AZUyuXO+hXky5ylD9D20mPtINW+u8NfqW6wTveAmQEpYfX03STGDJjStwjearcrG7HiGPjaSQFBaEt8ck5epml6P8WH9UgbXkO5u2YNk8DdmDvIWGvxPj7Ecugd9yuZrx64lzsxS4uDKn88wCQ9ffeLFeU5pkYnpaSdE42lVSN0dEUGN5+A2zQbc3w2AOsZzh/kAShUPMbcWV8ZBmT1JIfjTxeDh2dH/PeffjxhG46/J8GnJI+uwfQCnRD6JVTyWlQOnEv6lprk/csPEke60wT8rFZagYFUuySd9Rndc/bgUjxIhcETQFxqA52Eg7e+fycp3QKieen47UFHB9eIkDUMDcVZup8SD6ea8RRkd7JaGiXFt6HR93bTTtfg16gXF3ev9yfee+Tf2xzhacfDxEkYL0ioALb4dHH59vXn8aT32+N89Gg4fx7FXucokrPfmBDDXWdkLOOkvyr+ygBI18NPgXwclMdn42o19HsvN/4dcT8KpGW6hHEw1MY4dpMtF4ulg2Imm92ogjzSXe7GvgG1/sJylJBqddWnbD4cGfW5A4f90SF586P/AVBLAwQUAAAACACJaWNdM3vYtngKAACiGQAAEwAAAG1hc3RlcjkvdmFsaWRhdGUucHmNWVtv27gSfs+vIHoeLLm0YrlN2sTVAgd7QfvSE2QvL0Yg0BYda1eWZFJS7Cz2v59vSOriS4sN0MQmZ4bk8JtvZtg3b958llkyKeqKlZuDTlciY5XUlZ6zNE9kKfErr9ijSETNKiX+lKuqUKnULJeNVAyT+I2JNE/z5+DNmzdXa1VsWSmqTZYuWbotC1WxB3y1E4moxCoTWsOEmxQ6SVcVV7LMxEpeudE/dZHzKt12A3m9LQ8QZnnJsYeVMxhsi0RmnbGxGzVbagezQiRXV4lc005znVZpkXtGj+94w2teCsU30WfeyKxYpdUh+kVkWvr3Vww/6ZrlBdYvg1Svcc5KerXPCsXEUuPTD8HN21BOwtk9ltSS/SGyWv6sVKG8Uc3gWZ0m0rpIJiwptvgw8lvLm09TMrX54fM3jWBLuJUN3P5a5N83mFSHUka53Ff2eAEOJrYSd6Q93w/MtBHUkXFhUMlcY43FYnP9+emJW33zG/exq1MldfysRBK1nrHr7KpjffhGaKGUOHg7f/G1yOWRLX/efFuhuaRgVqmr013WTyd2ywgnbOfPDMC/7b7tVdLPS1ptmLUrc7HMpDmg59/vct7kPI6s5+CzrfY031W8qXhd8dLvTChZ1SpnuzxIJOC88fzAoNPzF9MnGLk0fHWydl606zprZtnjBYfqBr6qzj0AoKyr9nri57KOflN1C1Y7G1HAOUl/bpEOXQqniILheMqdOClqeMPz5yp/jnBBCBWAC2dZizqrYox6d+Gd9YIqXnT09z8WdNjYaiNXf3k5sIZYQnAnfDwG7lS60n7vetJakNBT9PfIyo3ul0WRefaL32v9My9VmlfWZK/H11mtN/bAxqxdeAQvxDUsxOtCxS0djdozL0aJbNKVHD1F0WhVJ2JE0Bh4kIFBJCOr3EpGZ5qcHN2N4svoye7gP2A3E2TYNsP20oSVqlinmWRwYE+ry6LOE6EO15DZs1WRa7PPSvMjqi0U+FGycLILjHlBV5GluS5Bjt6Uhzy8DQF8u0QsiSN0tHiak1n6+yKyDH+tdwqVSxUbFoDQQ88GWynyqObAaQT/bopI+Qy+YzU2w7xgOr3hwfSdHZPtWIix6a0dVGbw7mYa8DCcTgP/OwjvAECK2AypHm2tl6CfC0FtLa6B3er2vR8gV0hReZnMPeHz0D9Sf3F4hvSLd0QgYnHPh1xzbHUYcPfcRWy3pyN/B6Kk+/KMqkd54GXihRMBjt2KPajWN9dxLoa7xK2X8rVIE++FC38S3JA0XdoFo4tJ+ITpIdINwuJHGQNc8e+x29eI07rHm/Q/4bbe0UScF2orsvQVAWLmogvSHDYR8s8yWgRh+PHDDX8f3N08XVgcF6uztDQ72CJwYzqr3QGdw/9kMplBP43RrNlLeGM2QzJxl2k7rdONktpgt8ZKG3GPgKRmgvVhkyAskdmLPJNas2dV1OU9IpzqFaCNoi3VlYmvupoU64lNm0DhdpnmggoCbeMN0NNRHygBlROxkl7wkZuYCegYFDAAPVBvdLDLiO6LdINHOQk+2vFKHe4vmCJNqM7t9mytYe3sV7KsBsn/3okQMw3vAVfVpHpVoJQ5xN3hVocRx1YG7rfq5maJfUBRpXMpfvM0b+DrJB44IbYaMonsh9bhX35kZWELJ2O4vT4mKkaF4AFLZVJo2TuWwpsb4WUBTqDCKRXZRJdSJpYMW48Xhr9gsqUtoom94ZdJ8IEHHzoGoYlXOzHj4WDCzDStjmOq8IR5Gqs6xdztYIqufHDj3n4a7cePf/DXafQ6/ok3+N7seYOvzasfGJeJivIkakfDVO5o3oCGdiVvyuhitRnMbly9Oe3rzT6hMeeSczbYlZNdzy9QPRdpyknTiQwB424nbi8x3giVxF9+dLxRaBe0JgiPYmlIGBA7suoOTiRepchaXVQfm8fovzFPYh3erOUeZybclcSQrpeTbbpShZZAfuJKeoQ2N2W6YCUu0QX7WknJuhrYwo1cgXgF4oyVPoN20ENSPMl4LVhOkGKyZyU4EDIluAB57w1ejCYhhpDmdM/QMpelNiPYkdjb8NPXv/06wHru9oGkjzSLVDsDb3TI/Q7Ipg5i+RiLfANmu4I3RWTrjRj+Kr0zzU526Kpj2NnqRGTPAXG3Z1Ba+GMq8E15v0Ak/URJrLNlL+D7VgjIF6zAO/5lCBpaiX+tir/QsfQOtQC0KxoM3nRpaXgkM3Vn0NlB2MzE21hHAxNGpAsiJxKdmeODK6WCEUYWs7H3+Pl/D29vplN//PgwHs+uvbuxgRf/5lzod1Xmf9mmbdUfvnz9CjeiD6xdv071ZAXeRXdcuBLyumvXDywTS1S4Fv+26NmKvCa3wWse6rc7f67bXjjSYluiEXg/vbvloxUqXd/1jLrrEfoO56hNjD2DsPku2vXtRBM1/Zc6qvsvKPS6LzYVKHQOMV+K7LQLs1vjNuYtjpE/deRBdqI2yL2kSiJ98ef7Z42YZQDUwh3s9E6Zv5Tcccsavid6dfAil8PjMVptZXCG0zpgwQoVNTPeXkRMdvGv57C+PFnXWTZxRuxrCme2v0KE45Apgv1t3yvY7OkubAXi1tFRvfz2rJRHdxbUebqm6Omqd1Pe42xyX3pDAYq24tnwm8/7L7c4uO0FBrKo8E2B7xPLDScmwS0Pbn2XKY9mKDdjxubOY50uNRt6HM7ZtGybi5h4zxSjQKHrLKTpbuTrkKnbXoI81Cf0c6at3X0f7wVOQvE9J+RcZlHzMNSnZRtXPD7jzBM5eaHwp0XQxk+sKj6R48ahxIlxpm/Ih5082oDxT1b8mP06iMVl14XGBmCW+iSV3mHPevLV//Qu4BZT1D+ZT5bXXLkd15bRoGqGX0+GXwdZGqUfar6uFmUa6EXnzYo1o9AlSmfbYplmmOTsZUNdcZ3D/ysUgWJPKZ4ildL7Ot3LxFWE4WmWNZCZl7ML47dEN32iCGZ8GjjSpJeJXNZK9P0G1ei8HlR/p09Zu8W3O8T5916kjkUvPTedPfkQb+3sW8/3G94hqZ2+KSnio+jsmAizGclcn83M3MwQSH03sRXPuUQ1F7fXNuKER7PK5NYkSnRCbi62i5vfHCxjW4fboAXIL+Y5JKFyj3qFDJ7FihTXmnq4JXIZ1WzVRhqGPECU9lkCHAgukRy6pxS6bf67BYlvUbITvBFDQBB1ga78QeCL8EJkkx58wC+oQmP27zUML7ZQywBHT4QTMTuvWyhyh+52HqE2npowlYp8Ja2jjSHiB4Stidp+ECTw6Z0roddrqdDxSYrKTsIPqiJD6ev5g6cpU5kzatjh8kNeZImmt/SlVPfM1kssUeKZQpBe0AxgwamCAWwTbLBEpfGMGsY6nVZXsjRHH8+oVgFrh9cUn8MDukKMDMemYzJAcso4mflIjatUDcDTyL6HeJSRk+ufGhcjejNwL4ZPEb54atG+ID7171EkSx1aLXXrA/fUeT2yyVXH5r83AvrvhZEfvKi0khjaVx6NBAliC3cLMzw1tBrN/O4tmaB6vhv3Yv9onwbdm/1Dm8nFilp6ul62FgByQv+xoilQ2KUNuQ7FsgOWuvo/UEsBAhQDFAAAAAgASWljXT25J+dLAAAATgAAABMAAAAAAAAAAAAAAICBAAAAAG1hc3RlcjkvX19pbml0X18ucHlQSwECFAMUAAAACAC5aWNde+IGes8FAAC3DAAAEgAAAAAAAAAAAAAAgIF8AAAAbWFzdGVyOS9jb250cm9sLnB5UEsBAhQDFAAAAAgAzmljXRshiljwAwAAQAkAABEAAAAAAAAAAAAAAICBewYAAG1hc3RlcjkvbW9kZWxzLnB5UEsBAhQDFAAAAAgAiWljXZPiKpqfBQAAAwwAABIAAAAAAAAAAAAAAICBmgoAAG1hc3RlcjkvcGh5c2ljcy5weVBLAQIUAxQAAAAIAM5pY12uc9ofrgUAALkNAAAQAAAAAAAAAAAAAACAgWkQAABtYXN0ZXI5L3RyYWluLnB5UEsBAhQDFAAAAAgAiWljXTN72LZ4CgAAohkAABMAAAAAAAAAAAAAAICBRRYAAG1hc3RlcjkvdmFsaWRhdGUucHlQSwUGAAAAAAYABgB/AQAA7iAAAAAA'))) as z:z.extractall(ROOT)
sys.path.insert(0,str(ROOT))
OUTPUT=ROOT/'outputs'
print('GPU:',torch.cuda.get_device_name(0))


## آموزش GPU از صفر؛ فقط باقیماندهٔ معادلات
تعداد تکرارها و زمان واقعی در خروجی ثبت می‌شوند. آزمون‌ها از seed جدا و حل عددی مستقل استفاده می‌کنند.

In [ ]:
from master9.train import train
model,runtime=train(OUTPUT,flow_steps=6000,particle_steps=15000,batch=2048,require_gpu=True)
print(json.dumps(runtime,indent=2))


## آزمون فیزیکی برای هر ورودی
شامل رینولدز و ویسکوزیتهٔ سازگار، پروفایل و دبی سیال، محل رهایش، سرعت اولیه و میرایی آن، تغییر چگالی، اعتبار درگ استوکس، باقیماندهٔ مستقل و مقایسه با حل stiff Radau است. شکست هر معیار اجرای موفق را متوقف می‌کند؛ آستانه‌ها از قبل در کد تعیین شده‌اند.

In [ ]:
from master9.validate import run
physics_report=run(OUTPUT,require_gpu=True)
print(json.dumps(physics_report,indent=2))


## دنبال‌کردن مسیر با پارامترهای جدید
کنترل به u=±0.5 محدود است. فرمان‌های حاصل در حل مستقل اینرسی‌دار بازپخش می‌شوند. چهار آزمایش شامل سرعت سیال و ویسکوزیتهٔ مرزی و رهایش متغیر است.

In [ ]:
from master9.train import load
from master9.control import demos
model,_=load(OUTPUT)
model=model.double()
control_report=demos(model,OUTPUT)
from IPython.display import Image,display
display(Image(filename=str(OUTPUT/'control_results.png')))


## استفاده با پارامتر و مسیر خود
در مثال زیر اعداد SI هستند. `from_re` ویسکوزیته را سازگار می‌سازد. برای مسیر دلخواه تابع `target(z)` در `master9/control.py` قابل تغییر است؛ z و x داخل کنترل بی‌بعدند. خروجی u جریان واقعی سیم‌پیچ نیست؛ مدل کالیبراسیون سخت‌افزار لازم است.

In [ ]:
from master9.physics import Parameters
p=Parameters.from_re(.8,mean=.015,rho=1000,x0=0,z0=-.08,vx0=.0005,vz0=.02)
print('Parameters:',p)
print('Re:',p.Re,'relaxation_s:',p.relaxation_s)
print('OUTPUTS:',OUTPUT)
print('ALL_PHYSICS_TESTS_PASSED:',physics_report['all_passed'])
print('ALL_CONTROL_TESTS_PASSED:',control_report['all_passed'])
